# YetiRankを用いてcatboost(3)

In [1]:
import pandas as pd, numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from lightgbm import LGBMClassifier

import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

print("ok")

ok


In [4]:
# -------------------------
# 1. データ読み込み & サンプリング
# -------------------------
transactions = pd.read_csv("/Users/kurokawa/Desktop/kaggle/H&M/transactions_train.csv")
customers = pd.read_csv("/Users/kurokawa/Desktop/kaggle/H&M/customers.csv")
articles = pd.read_csv("/Users/kurokawa/Desktop/kaggle/H&M/articles.csv")

print("transactions shape: ", transactions.shape)
print("customers shape: ", customers.shape)
print("articles shape: ", articles.shape)

articles.head()

transactions shape:  (31788324, 5)
customers shape:  (1371980, 7)
articles shape:  (105542, 25)


,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,4,Dark,5,Black,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,3,Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,1,Dusty Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,4,Dark,5,Black,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulded, lightly padded cups that shape the bust and pro..."
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,3,Light,9,White,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulded, lightly padded cups that shape the bust and pro..."


In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostRanker, Pool

# ------------------------------
# 0. データ読み込み
# ------------------------------
transactions = pd.read_csv('/Users/kurokawa/Desktop/kaggle/H&M/transactions_train.csv')
customers = pd.read_csv('/Users/kurokawa/Desktop/kaggle/H&M/customers.csv')
articles = pd.read_csv('/Users/kurokawa/Desktop/kaggle/H&M/articles.csv')
sample_submission = pd.read_csv('/Users/kurokawa/Desktop/kaggle/H&M/sample_submission.csv')

# ------------------------------
# 1. 正例（直近の購入）と負例（5倍）
# ------------------------------
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])
positive = transactions[transactions['t_dat'] >= '2020-09-15'][['customer_id', 'article_id']].drop_duplicates()
positive['target'] = 1

np.random.seed(42)
neg = pd.DataFrame({
    'customer_id': np.random.choice(positive['customer_id'].unique(), size=len(positive)*5),
    'article_id': np.random.choice(positive['article_id'].unique(), size=len(positive)*5),
})
neg['target'] = 0

neg = neg[~neg.set_index(['customer_id', 'article_id']).index.isin(
    positive.set_index(['customer_id', 'article_id']).index
)]

train_df = pd.concat([positive, neg], ignore_index=True)
train_df['group_id'] = train_df['customer_id']

# ------------------------------
# 2. 特徴量追加
# ------------------------------
# 顧客特徴量
customer_features = customers[['customer_id', 'age', 'fashion_news_frequency']].copy()
# 年齢を3分割：<30, 30–59, 60+
customer_features['age_group3'] = pd.cut(
    customer_features['age'],
    bins=[0, 30, 60, 120],  # 年齢の最大値を想定して上限120に
    labels=['young', 'middle', 'senior'],
    right=False  # 30歳をmiddleに含める（[0,30), [30,60), [60,120)）
)

# 商品特徴量
articles_features = articles[['article_id', 'product_type_no', 'index_group_no', 'garment_group_no']].copy()

# 過去購入履歴（再購買フラグ）
purchase_history = transactions[['customer_id', 'article_id']].drop_duplicates()
purchase_history['past_purchased'] = 1

# 結合
train_df = train_df.merge(customer_features, on='customer_id', how='left')
train_df = train_df.merge(articles_features, on='article_id', how='left')
train_df = train_df.merge(purchase_history, on=['customer_id', 'article_id'], how='left')
train_df['past_purchased'] = train_df['past_purchased'].fillna(0)

# 説明変数とカテゴリ変数
feature_cols = ['product_type_no', 'index_group_no', 'garment_group_no', 'age', 'past_purchased']
cat_cols = ['product_type_no', 'index_group_no', 'garment_group_no']

# ------------------------------
# 3. 学習
# ------------------------------
train_df = train_df.sort_values('group_id').reset_index(drop=True)
train_pool = Pool(train_df[feature_cols], label=train_df['target'], group_id=train_df['group_id'], cat_features=cat_cols)

model = CatBoostRanker(
    iterations=300,
    learning_rate=0.1,
    loss_function='YetiRank',
    eval_metric='NDCG',
    verbose=50
)

model.fit(train_pool)

# ------------------------------
# 4. 推薦対象作成（人気TOP 3500商品）
# ------------------------------
popular_articles_3500 = (
    transactions['article_id']
    .value_counts()
    .head(3500)
    .index
    .tolist()
)

test_customers = train_df['customer_id'].unique()
test_articles = popular_articles_3500

test_df = pd.DataFrame([(c, a) for c in test_customers for a in test_articles], columns=['customer_id', 'article_id'])

# 特徴量結合
test_df = test_df.merge(customer_features, on='customer_id', how='left')
test_df = test_df.merge(articles_features, on='article_id', how='left')
test_df = test_df.merge(purchase_history, on=['customer_id', 'article_id'], how='left')
test_df['past_purchased'] = test_df['past_purchased'].fillna(0)

# ------------------------------
# 5. スコア予測と上位12件抽出
# ------------------------------
test_pool = Pool(test_df[feature_cols], cat_features=cat_cols)
test_df['score'] = model.predict(test_pool)

recommendations = test_df.sort_values(['customer_id', 'score'], ascending=[True, False]) \
                         .groupby('customer_id')['article_id'] \
                         .apply(lambda x: [str(a).zfill(10) for a in x.head(12)])

# ------------------------------
# 6. 補完処理（四半期 × 年齢層）
# ------------------------------
# 全体人気TOP12（補完用）
popular_articles = [str(a).zfill(10) for a in transactions['article_id'].value_counts().head(12).index.tolist()]

# 年齢層と四半期
latest_dates = transactions.groupby('customer_id')['t_dat'].max().reset_index()
latest_dates['t_dat'] = pd.to_datetime(latest_dates['t_dat'])
unknown_customers = pd.DataFrame({'customer_id': test_customers})
unknown_customers = unknown_customers.merge(customer_features[['customer_id', 'age_group3']], on='customer_id', how='left')
unknown_customers = unknown_customers.merge(latest_dates, on='customer_id', how='left')
unknown_customers["quarter"] = pd.PeriodIndex(unknown_customers["t_dat"], freq="Q").astype(str)

# 四半期×年齢層の人気商品
transactions['quarter'] = pd.PeriodIndex(transactions['t_dat'], freq='Q').astype(str)
transactions = transactions.merge(customer_features[['customer_id', 'age_group3']], on='customer_id', how='left')
top_items_dict = (
    transactions.groupby(['quarter', 'age_group3'])['article_id']
    .value_counts()
    .groupby(['quarter', 'age_group3'])
    .head(12)
    .reset_index()
    .groupby(['quarter', 'age_group3'])['article_id']
    .apply(lambda x: [str(a).zfill(10) for a in x])
    .to_dict()
)

# 補完関数
def recommend_12_articles_with_context(article_list, quarter, age_group):
    if len(article_list) >= 12:
        return article_list[:12]
    else:
        needed = 12 - len(article_list)
        key = (quarter, age_group)
        if key in top_items_dict:
            fill_items = [a for a in top_items_dict[key] if a not in article_list]
        else:
            fill_items = [a for a in popular_articles if a not in article_list]
        return article_list + fill_items[:needed]

# 適用
merged_pred = recommendations.reset_index().merge(
    unknown_customers[['customer_id', 'quarter', 'age_group3']],
    on='customer_id', how='left'
)

merged_pred['prediction'] = merged_pred.apply(
    lambda row: recommend_12_articles_with_context(row['article_id'], row['quarter'], row['age_group3']),
    axis=1
)

# ------------------------------
# 7. 提出用に整形・保存
# ------------------------------
merged_pred['prediction'] = merged_pred['prediction'].apply(lambda x: ' '.join(x))
submission = sample_submission[['customer_id']].merge(merged_pred[['customer_id', 'prediction']], on='customer_id', how='left')
fallback_12 = ' '.join(popular_articles[:12])
submission['prediction'] = submission['prediction'].fillna(fallback_12)

submission.to_csv("YetiRank_catboost_submission.csv", index=False)
print("✅ CSVを出力しました")


Groupwise loss function. OneHotMaxSize set to 10
0:	total: 408ms	remaining: 2m 2s
50:	total: 16.7s	remaining: 1m 21s
100:	total: 31.9s	remaining: 1m 2s
150:	total: 47.7s	remaining: 47.1s
200:	total: 1m 3s	remaining: 31.5s
250:	total: 1m 20s	remaining: 15.8s
299:	total: 1m 37s	remaining: 0us


In [5]:
# 顧客側特徴量（例）
customer_features = customers.copy()

# 商品側特徴量（例）
articles_features = articles.copy()

# それらをトランザクションに結合
merged = transactions.merge(customer_features, on="customer_id", how="left")
merged = merged.merge(articles_features, on="article_id", how="left")
print("merged shape: ", merged.shape)

merged shape:  (31788324, 35)


In [6]:
# 正例：直近X日（例：2020-09-15以降）の購入
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])
positive = transactions[transactions['t_dat'] >= '2020-09-15'][['customer_id', 'article_id']].drop_duplicates()
positive['target'] = 1

# 負例：正例の顧客・商品からランダムに5倍サンプリング（かつ正例と重複しない）
import numpy as np

np.random.seed(42)
neg = pd.DataFrame({
    'customer_id': np.random.choice(positive['customer_id'].unique(), size=len(positive)*5),
    'article_id': np.random.choice(positive['article_id'].unique(), size=len(positive)*5),
})
neg['target'] = 0

# 正例と重複しているものを除く
neg = neg[~neg.set_index(['customer_id', 'article_id']).index.isin(
    positive.set_index(['customer_id', 'article_id']).index
)]

# 結合
train_df = pd.concat([positive, neg], ignore_index=True)
train_df['group_id'] = train_df['customer_id']


In [7]:
# 顧客側特徴量
customer_features = customers[['customer_id', 'age', 'fashion_news_frequency']].copy()
customer_features['age_group3'] = pd.cut(customer_features['age'], bins=[0, 25, 50, 100], labels=['young', 'middle', 'senior'])

# 商品側特徴量
articles_features = articles[['article_id', 'product_type_no', 'index_group_no', 'garment_group_no']].copy()

# 顧客 × 商品 のインタラクション特徴量（再購買フラグ）
purchase_history = transactions[['customer_id', 'article_id']].drop_duplicates()
purchase_history['past_purchased'] = 1

# 特徴量結合
train_df = train_df.merge(customer_features, on='customer_id', how='left')
train_df = train_df.merge(articles_features, on='article_id', how='left')
train_df = train_df.merge(purchase_history, on=['customer_id', 'article_id'], how='left')
train_df['past_purchased'] = train_df['past_purchased'].fillna(0)


In [8]:
train_df.head()

,customer_id,article_id,target,group_id,age,fashion_news_frequency,age_group3,product_type_no,index_group_no,garment_group_no,past_purchased
0,000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318,794321007,1,000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318,24.0,NONE,young,262,26,1001,1.0
1,00040239317e877c77ac6e79df42eb2633ad38fcac09fc0094e549180ddc201c,875272011,1,00040239317e877c77ac6e79df42eb2633ad38fcac09fc0094e549180ddc201c,50.0,NONE,middle,259,2,1016,1.0
2,00040239317e877c77ac6e79df42eb2633ad38fcac09fc0094e549180ddc201c,875272012,1,00040239317e877c77ac6e79df42eb2633ad38fcac09fc0094e549180ddc201c,50.0,NONE,middle,259,2,1016,1.0
3,000749135ee9aa3a24c2316ea5ae4f495b39c1653c5612bb5b239f1b2a182a2a,800691007,1,000749135ee9aa3a24c2316ea5ae4f495b39c1653c5612bb5b239f1b2a182a2a,29.0,NONE,middle,255,1,1002,1.0
4,000749135ee9aa3a24c2316ea5ae4f495b39c1653c5612bb5b239f1b2a182a2a,800691008,1,000749135ee9aa3a24c2316ea5ae4f495b39c1653c5612bb5b239f1b2a182a2a,29.0,NONE,middle,255,1,1002,1.0


In [ ]:
# 人気商品TOP 3500件を候補に使う
popular_articles_3500 = (
    transactions['article_id']
    .value_counts()
    .head(3500)
    .index
    .tolist()
)

test_articles = popular_articles_3500
test_customers = train_df['customer_id'].unique()

# 全組み合わせ（メモリ注意）
test_df = pd.DataFrame([(c, a) for c in test_customers for a in test_articles],
                       columns=['customer_id', 'article_id'])
